In [1]:
# Cell 1 — Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.io import save_comparisons

In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)

print(f"Config: {config_path}")

Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\notebook_config.yaml


In [3]:
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}

runner = VideoPipelineRunner.build(config, models)

print(f"ROI model:   {type(roi_model).__name__}")
print(f"Spike model: {type(spike_model).__name__}")

ROI model:   RandomForestClassifier
Spike model: LogisticRegression


In [4]:
EXPERIMENT_ROOT = Path(r"E:\GCaMP6s_EX357")  # TODO: change per experiment
assert EXPERIMENT_ROOT.exists(), f"Experiment root not found: {EXPERIMENT_ROOT}"

builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(EXPERIMENT_ROOT)
print_tree(tree)

└── GCaMP6s_EX357
    ├── GCaMP6s_EX357_4Conditions
    │   ├── Week 1
    │   │   ├── 1-1-1
    │   │   ├── 1-1-2_0001
    │   │   ├── 1-1-3
    │   │   ├── 1-2-1
    │   │   ├── 1-2-2
    │   │   ├── 1-2-3
    │   │   ├── 1-3-1
    │   │   ├── 1-3-2
    │   │   ├── 1-3-3
    │   │   ├── 1-4-1
    │   │   ├── 1-4-2
    │   │   └── 1-4-3
    │   └── Week 2
    │       ├── 2-1-1
    │       ├── 2-1-2
    │       ├── 2-1-3
    │       ├── 2-2-1
    │       ├── 2-2-2
    │       ├── 2-2-3
    │       ├── 2-3-1
    │       ├── 2-3-2
    │       ├── 2-3-3
    │       ├── 2-4-1
    │       ├── 2-4-2
    │       └── 2-4-3
    ├── GCaMP6s_EX357_DL-AP5
    │   ├── Week 2 25uM
    │   │   ├── 2-1
    │   │   ├── 2-1_25uM_8m
    │   │   ├── 2-2
    │   │   ├── 2-2_25uM_2m
    │   │   ├── 2-3
    │   │   └── 2-3_25uM_14m
    │   └── Week 2 50uM
    │       ├── 2-1
    │       ├── 2-1_50uM_2m
    │       ├── 2-2
    │       ├── 2-2_50uM_8m
    │       ├── 2-3
    │       └── 2-3_50uM_14m
    └── GC

In [ ]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
)
processor.process_tree(tree, verbose=True)


 Processing: 1-1-1
  Traces: 372 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 302/372 kept (81.2%)
  Spikes: 3901/41280 kept | neurons 302 → 300
  Grouping (corr): 2 groups | agreement=0.00

 Processing: 1-1-2_0001
  Traces: 469 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 347/469 kept (74.0%)
  Spikes: 4236/49839 kept | neurons 347 → 344
  Grouping (corr): 0 groups | agreement=0.00

 Processing: 1-1-3
  Traces: 480 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 397/480 kept (82.7%)
  Spikes: 6088/52700 kept | neurons 397 → 396
  Grouping (corr): 3 groups | agreement=0.00

 Processing: 1-2-1
  Traces: 378 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 330/378 kept (87.3%)
  Spikes: 6016/42712 kept | neurons 330 → 330
  Grouping (corr): 7 groups | agreement=0.00

 Processing: 1-2-2
  Traces: 603 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 494/603 kept (81.9%)
  Spikes: 8211/65553 kept | neurons 494 → 494
  Grouping (corr): 2 groups | agreement=0.00

 Processing: 1-2-3
  Traces: 512 ROIs, 3650 frames @ 1

In [6]:
sibling_tables = processor.compare_siblings(tree)

for node_path, df in sibling_tables.items():
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


Node: C:\Users\mzinn1\Desktop\Morgan 1-20-26
               child  n_videos  n_neurons  n_groups  decay_tau_mean_unweighted  decay_tau_var_unweighted  decay_tau_within_unweighted  decay_tau_between_unweighted  half_max_width_mean_unweighted  half_max_width_var_unweighted  half_max_width_within_unweighted  half_max_width_between_unweighted  rise_slope_mean_unweighted  rise_slope_var_unweighted  rise_slope_within_unweighted  rise_slope_between_unweighted  decay_tau_mean_weighted  decay_tau_var_weighted  decay_tau_within_weighted  decay_tau_between_weighted  half_max_width_mean_weighted  half_max_width_var_weighted  half_max_width_within_weighted  half_max_width_between_weighted  rise_slope_mean_weighted  rise_slope_var_weighted  rise_slope_within_weighted  rise_slope_between_weighted  spike_frequency_mean_unweighted  spike_frequency_var_unweighted  spike_frequency_within_unweighted  spike_frequency_between_unweighted  spike_frequency_mean_weighted  spike_frequency_var_weighted  spike_fr

In [ ]:
save_comparisons(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)

: 